# Install and init DVC

Prerequisites: 
-  DVC and requirements.txt packages installed (if not - check README.md file for instructions)
-  A project repository is a Git repo 

## Install with pip

In [1]:
!pip install dvc==2.0.5

  Using cached dvc-2.0.5-py2.py3-none-any.whl (622 kB)
  Using cached voluptuous-0.12.1-py3-none-any.whl (29 kB)
  Using cached shtab-1.3.5-py2.py3-none-any.whl (12 kB)
  Using cached fsspec-0.8.7-py3-none-any.whl (103 kB)
  Using cached grandalf-0.6-py3-none-any.whl (31 kB)
  Using cached colorama-0.4.4-py2.py3-none-any.whl (16 kB)
  Using cached networkx-2.5-py3-none-any.whl (1.6 MB)
  Using cached toml-0.10.2-py2.py3-none-any.whl (16 kB)
  Using cached psutil-5.8.0-cp38-cp38-manylinux2010_x86_64.whl (296 kB)
  Using cached diskcache-5.2.1-py3-none-any.whl (44 kB)
Processing /home/alex/.cache/pip/wheels/34/2a/24/a490264ae9041fd48f778ff393526572c80bb498ddecb07ea5/configobj-5.0.6-py3-none-any.whl
Processing /home/alex/.cache/pip/wheels/f4/f4/20/d7114a2d97a2e72ce7189a7f63a6fbd2a33ec6ae6010befca5/flufl.lock-3.2-py3-none-any.whl
  Using cached flatten_dict-0.3.0-py2.py3-none-any.whl (8.2 kB)
Processing /home/alex/.cache/pip/wheels/bc/7d/26/d3e5714967aa183b5c659493fa88daf270295948d0ac27768

## Checkout branch `experiments`

In [2]:
!git checkout -b experiments

Переключено на новую ветку «experiments»


## Initialize DVC

References: 
- https://dvc.org/doc/get-started/initialize 

In [3]:
!dvc init

Initialized DVC repository.

You can now commit the changes to git.

+---------------------------------------------------------------------+
|                                                                     |
|        DVC has enabled anonymous aggregate usage analytics.         |
|     Read the analytics documentation (and how to opt-out) here:     |
|             <https://dvc.org/doc/user-guide/analytics>              |
|                                                                     |
+---------------------------------------------------------------------+

What's next?
------------
- Check out the documentation: <https://dvc.org/doc>
- Get help and share ideas: <https://dvc.org/chat>
- Star us on GitHub: <https://github.com/iterative/dvc>


## Commit changes

In [4]:
%%bash

git add .
git commit -m "Initialize DVC"

[experiments a270f00] Initialize DVC
 11 files changed, 638 insertions(+), 57 deletions(-)
 create mode 100644 .dvc/.gitignore
 create mode 100644 .dvc/config
 create mode 100644 .dvc/plots/confusion.json
 create mode 100644 .dvc/plots/confusion_normalized.json
 create mode 100644 .dvc/plots/default.json
 create mode 100644 .dvc/plots/linear.json
 create mode 100644 .dvc/plots/scatter.json
 create mode 100644 .dvc/plots/smooth.json
 create mode 100644 .dvcignore


## Review Files and Directories created by DVC

In [5]:
!ls -a .dvc 

.  ..  config  .gitignore  plots  tmp


In [6]:
!cat .dvc/.gitignore

/config.local
/tmp
/cache


# Quick Tour of DVC features

## Data Versioning

In [7]:
# Get data 

import pandas as pd
from sklearn.datasets import load_iris

data = load_iris(as_frame=True)
list(data.target_names)
data.frame.to_csv('data/iris.csv', index=False)

In [8]:
# Look on data

data.frame.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [9]:
%%bash

du -sh data/*

4,0K	data/iris.csv


## Add file under DVC control

In [17]:
%%bash

dvc add data/iris.csv


To track the changes with git, run:

	git add data/.gitignore data/iris.csv.dvc


In [18]:
!du -sh data/*

4,0K	data/iris.csv
4,0K	data/iris.csv.dvc


In [19]:
!git status -s data/

?? data/.gitignore
?? data/iris.csv.dvc


In [20]:
%%bash

git add .
git commit -m "Add a source dataset"

[experiments 9d48823] Add a source dataset
 4 files changed, 25 insertions(+), 39 deletions(-)
 create mode 100644 data/.gitignore
 create mode 100644 data/iris.csv.dvc


### What is DVC-file?

Data file internals


>    If you take a look at the DVC-file, you will see that only outputs are defined in outs. 
    In this file, only one output is defined. The output contains the data file path in the repository and md5 cache.
    This md5 cache determines a location of the actual content file in DVC cache directory .dvc/cache
    >> Output from DVC-files defines the relationship between the data file path in a repository and the path in a cache directory. See also DVC File Format



(c) dvc.org https://dvc.org/doc/tutorial/define-ml-pipeline

In [21]:
!cat data/iris.csv.dvc

outs:
- md5: 4d301abed5efe50eccda350cde38e0eb
  size: 2777
  path: iris.csv


## Create and Reproduce ML pipelines 

Stages 
- extract features 
- split dataset 
- train 
- evaluate 


### Add a pipeline stage with 'dvc stage add'

In [32]:
!dvc stage add \
    -n feature_extraction \
    -d src/featurization.py \
    -d data/iris.csv \
    -o data/iris_featurized.csv \
    python src/featurization.py

Creating 'dvc.yaml'                                                   core>
Adding stage 'feature_extraction' in 'dvc.yaml'

To track the changes with git, run:

	git add dvc.yaml


In [33]:
!cat dvc.yaml

stages:
  feature_extraction:
    cmd: python src/featurization.py
    deps:
    - data/iris.csv
    - src/featurization.py
    outs:
    - data/iris_featurized.csv


### Add split train/test stage

In [34]:
!dvc stage add \
    -n split_dataset \
    -d src/split_dataset.py \
    -d data/iris_featurized.csv \
    -o data/train.csv \
    -o data/test.csv \
    python src/split_dataset.py --test_size 0.4

Adding stage 'split_dataset' in 'dvc.yaml'                            core>

To track the changes with git, run:

	git add dvc.yaml


In [35]:
!cat dvc.yaml

stages:
  feature_extraction:
    cmd: python src/featurization.py
    deps:
    - data/iris.csv
    - src/featurization.py
    outs:
    - data/iris_featurized.csv
  split_dataset:
    cmd: python src/split_dataset.py --test_size 0.4
    deps:
    - data/iris_featurized.csv
    - src/split_dataset.py
    outs:
    - data/test.csv
    - data/train.csv


### Add train stage

In [36]:
!dvc stage add \
    -n train \
    -d src/train.py \
    -d data/train.csv \
    -o data/model.joblib \
    python src/train.py

Adding stage 'train' in 'dvc.yaml'                                    core>

To track the changes with git, run:

	git add dvc.yaml


In [37]:
!cat dvc.yaml

stages:
  feature_extraction:
    cmd: python src/featurization.py
    deps:
    - data/iris.csv
    - src/featurization.py
    outs:
    - data/iris_featurized.csv
  split_dataset:
    cmd: python src/split_dataset.py --test_size 0.4
    deps:
    - data/iris_featurized.csv
    - src/split_dataset.py
    outs:
    - data/test.csv
    - data/train.csv
  train:
    cmd: python src/train.py
    deps:
    - data/train.csv
    - src/train.py
    outs:
    - data/model.joblib


### Add evaluate stage

In [38]:
!dvc stage add \
    -n evaluate \
    -d src/train.py \
    -d src/evaluate.py \
    -d data/test.csv \
    -d data/model.joblib \
    -m data/eval.txt \
    python src/evaluate.py

Adding stage 'evaluate' in 'dvc.yaml'                                 core>

To track the changes with git, run:

	git add data/.gitignore dvc.yaml


In [39]:
!cat dvc.yaml

stages:
  feature_extraction:
    cmd: python src/featurization.py
    deps:
    - data/iris.csv
    - src/featurization.py
    outs:
    - data/iris_featurized.csv
  split_dataset:
    cmd: python src/split_dataset.py --test_size 0.4
    deps:
    - data/iris_featurized.csv
    - src/split_dataset.py
    outs:
    - data/test.csv
    - data/train.csv
  train:
    cmd: python src/train.py
    deps:
    - data/train.csv
    - src/train.py
    outs:
    - data/model.joblib
  evaluate:
    cmd: python src/evaluate.py
    deps:
    - data/model.joblib
    - data/test.csv
    - src/evaluate.py
    - src/train.py
    metrics:
    - data/eval.txt


### Run DVC pipeline (all stages)

In [42]:
!dvc repro

'data/iris.csv.dvc' didn't change, skipping                           core>
Running stage 'feature_extraction':
> python src/featurization.py
Generating lock file 'dvc.lock'                                                 
Updating lock file 'dvc.lock'

Running stage 'split_dataset':
> python src/split_dataset.py --test_size 0.4
Updating lock file 'dvc.lock'                                         core>

Running stage 'train':
> python src/train.py
Updating lock file 'dvc.lock'                                         core>

Running stage 'evaluate':
> python src/evaluate.py
Updating lock file 'dvc.lock'                                         core>

To track the changes with git, run:

	git add dvc.lock
Use `dvc push` to send your updates to remote storage.


In [43]:
!ls 

data			 dvc.lock  dvc.yaml   requirements.txt
dvc-1-get-started.ipynb  dvc-venv  README.md  src


In [44]:
import pandas as pd

features = pd.read_csv('data/iris_featurized.csv')
features.head()

,sepal_length,sepal_width,petal_length,petal_width,target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [45]:
!git status -s

 M data/.gitignore
 M dvc-1-get-started.ipynb
?? dvc.lock
?? dvc.yaml


In [46]:
%%bash
git add .
git commit -m "Create DVC pipeline"

[experiments 2bdd8e7] Create DVC pipeline
 4 files changed, 378 insertions(+), 219 deletions(-)
 create mode 100644 dvc.lock
 create mode 100644 dvc.yaml


### Reproduce pipeline

In [47]:
!dvc repro split_dataset

'data/iris.csv.dvc' didn't change, skipping                           core>
Stage 'feature_extraction' didn't change, skipping
Stage 'split_dataset' didn't change, skipping
Data and pipelines are up to date.


## Collaborate on ML Experiments 

### Specify remote storage (local ~ /tmp/dvc)


In [48]:
!dvc remote add -d local /tmp/dvc

Setting 'local' as a default remote.


### Push features to remote storage

In [49]:
!dvc push

  0% Uploading|                                      |0/6 [00:00<?,     ?file/s]
!
  0%|          |data/train.csv                 0.00/1.68k [00:00<?,       ?it/s]

  0%|          |data/test.csv                  0.00/1.14k [00:00<?,       ?it/s]


  0%|          |data/iris.csv                  0.00/2.78k [00:00<?,       ?it/s]



  0%|          |data/model.joblib                   0/897 [00:00<?,       ?it/s]




  0%|          |data/eval.txt                       0/124 [00:00<?,       ?it/s]
                                                                                

                                                                                
  0%|          |data/iris_featurized.csv       0.00/2.76k [00:00<?,       ?it/s]


                                                                                



                                                                                




                                                                                
6 file

### Create tag `experiment-1`

In [50]:
!git tag -a experiment-1 -m "experiment-1"

### Checkout into your teammate experiment state

In [51]:
%%bash 

git checkout experiment-1
dvc checkout

M	.dvc/config
M	dvc-1-get-started.ipynb


Note: switching to 'experiment-1'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD сейчас на 2bdd8e7 Create DVC pipeline


### Check Metrics

In [52]:
!dvc metrics show

Path           f1_score                                               core>
data/eval.txt  0.78618


### Reproduce experiment

In [53]:
# Nothing to reproduce
!dvc repro

'data/iris.csv.dvc' didn't change, skipping                           core>
Stage 'feature_extraction' didn't change, skipping
Stage 'split_dataset' didn't change, skipping
Stage 'train' didn't change, skipping
Stage 'evaluate' didn't change, skipping
Data and pipelines are up to date.


In [54]:
!dvc repro -f

Verifying data sources in stage: 'data/iris.csv.dvc'                  core>
                                                                                
Running stage 'feature_extraction':
> python src/featurization.py
                                                                      core>
Running stage 'split_dataset':
> python src/split_dataset.py --test_size 0.4
                                                                      core>
Running stage 'train':
> python src/train.py
                                                                      core>
Running stage 'evaluate':
> python src/evaluate.py
Use `dvc push` to send your updates to remote storage.                core>


In [55]:
# Check Metrics

!dvc metrics show

Path           f1_score                                               core>
data/eval.txt  0.78618
